# FarmShare GPU Test

Quick smoke test to verify your GPU environment is working.

In [ ]:
import torch

print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"GPUs:     {torch.cuda.device_count()}")
else:
    print("No GPU detected. Are you on an oat node?")

In [ ]:
# Quick matrix multiply on GPU
if torch.cuda.is_available():
    x = torch.randn(4096, 4096, device="cuda")
    y = torch.randn(4096, 4096, device="cuda")
    
    torch.cuda.synchronize()
    import time
    start = time.time()
    z = torch.matmul(x, y)
    torch.cuda.synchronize()
    elapsed = time.time() - start
    
    print(f"4096x4096 matmul: {elapsed*1000:.1f} ms")
    print(f"Result shape: {z.shape}")
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Check other installed packages
import numpy as np
import pandas as pd
import sklearn
import matplotlib

print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")

In [ ]:
# Install new packages with %pip (no restart needed for most packages)
# %pip install transformers

## Training with Checkpoints

SLURM jobs have a 48-hour time limit. For training runs that might take longer, save checkpoints periodically so you can resume from where you left off in the next job.

Key points:
- Save checkpoints to your **home directory** (`~/checkpoints/`) or **scratch** (`/scratch/users/yoursunetid/checkpoints/`). Both are on shared NFS, so they persist after the job ends.
- Save model weights, optimizer state, epoch number, and loss so training resumes cleanly.
- Check for an existing checkpoint at startup to auto-resume.

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

# --- Configuration ---
CHECKPOINT_DIR = Path.home() / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "model_checkpoint.pt"

# --- Example model ---
model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

if torch.cuda.is_available():
    model = model.cuda()

# --- Resume from checkpoint if one exists ---
start_epoch = 0
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resumed from checkpoint at epoch {start_epoch}")
else:
    print("No checkpoint found, starting from scratch")

# --- Training loop with checkpointing ---
NUM_EPOCHS = 100
SAVE_EVERY = 10  # save checkpoint every N epochs

for epoch in range(start_epoch, NUM_EPOCHS):
    # ... your training code here ...
    loss = 0.0  # placeholder

    # Save checkpoint periodically
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": loss,
        }, CHECKPOINT_PATH)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - checkpoint saved")

print(f"Training complete. Final checkpoint: {CHECKPOINT_PATH}")